In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

def main():
    # -----------------------------
    # 1. 共通設定パラメータ
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 8
    NUM_EPOCHS = 30
    LEARNING_RATE = 0.001
    NUM_CLASSES = 3
    K_FOLDS = 6 # 6分割

    # =========================================================
    # ★ここに「試したいデータ総数のリスト」を指定してください
    # ループで順番に実行されます
    # =========================================================
    TARGET_TOTAL_SIZES = [18, 30, 42, 54, 66,72, 84, 96, 108, 120, 132, 144, 156] 
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # データセットの読み込み（共通部分）
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # フォルダ読み込みは重いのでループの外で一度だけ行う
    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    print(f"検知されたクラス: {class_names}") 

    # =========================================================
    # ★ループ開始：指定したデータサイズごとに実行
    # =========================================================
    for total_size_req in TARGET_TOTAL_SIZES:
        print(f"\n========================================================")
        print(f"★ 実験開始: 目標データ総数 {total_size_req}枚")
        print(f"========================================================")

        # パラメータ計算
        IMAGES_PER_CLASS = total_size_req // NUM_CLASSES
        ACTUAL_TOTAL = IMAGES_PER_CLASS * NUM_CLASSES
        SEED_VALUE = total_size_req # シード値をデータ数に合わせて変更（固定したい場合は定数に）

        # 保存先ディレクトリ（データ数ごとに分ける）
        SAVE_DIR = f'resnet18_cv_total{ACTUAL_TOTAL}'
        os.makedirs(SAVE_DIR, exist_ok=True)

        print(f"【設定確認】")
        print(f"  指定: {total_size_req}枚 -> 実使用: {ACTUAL_TOTAL}枚 (各クラス {IMAGES_PER_CLASS}枚)")
        print(f"  保存先: {SAVE_DIR}")

        # -----------------------------
        # 2. データセットの抽出 (サイズごとにやり直す)
        # -----------------------------
        used_indices = []
        used_labels = []

        np.random.seed(SEED_VALUE) # シード固定

        print("\n--- データセットの抽出 ---")
        for class_idx in range(NUM_CLASSES):
            indices_of_class = np.where(all_targets == class_idx)[0]
            
            if len(indices_of_class) < IMAGES_PER_CLASS:
                 print(f"警告: クラス {class_names[class_idx]} は {len(indices_of_class)}枚不足。あるだけ使用します。")
                 select_num = len(indices_of_class)
            else:
                 select_num = IMAGES_PER_CLASS

            np.random.shuffle(indices_of_class)
            selected_indices = indices_of_class[:select_num]
            
            used_indices.extend(selected_indices)
            used_labels.extend([class_idx] * len(selected_indices))
            
            # print(f"[{class_names[class_idx]}] 抽出: {len(selected_indices)}枚")

        used_indices = np.array(used_indices)
        used_labels = np.array(used_labels)
        
        print(f"実際に使用するデータ総数: {len(used_indices)}枚")

        # -----------------------------
        # 3. K-Fold Cross Validation ループ
        # -----------------------------
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

        for fold, (train_idx_in_subset, val_idx_in_subset) in enumerate(skf.split(used_indices, used_labels)):
            print(f"\n--- [Total: {ACTUAL_TOTAL}] Fold {fold + 1}/{K_FOLDS} ---")

            train_real_indices = used_indices[train_idx_in_subset]
            val_real_indices = used_indices[val_idx_in_subset]

            train_dataset = Subset(full_dataset, train_real_indices)
            val_dataset = Subset(full_dataset, val_real_indices)

            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

            # モデル初期化
            model = models.resnet18(weights=None)
            model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
            model = model.to(device)

            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

            history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

            # 学習ループ
            for epoch in range(NUM_EPOCHS):
                model.train()
                running_loss, correct, total = 0.0, 0, 0

                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

                epoch_loss = running_loss / len(train_dataset)
                epoch_acc = correct / total

                model.eval()
                val_loss, val_correct, val_total = 0.0, 0, 0
                with torch.no_grad():
                    for inputs, labels in val_loader:
                        inputs, labels = inputs.to(device), labels.to(device)
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                        val_loss += loss.item() * inputs.size(0)
                        _, predicted = torch.max(outputs, 1)
                        val_total += labels.size(0)
                        val_correct += (predicted == labels).sum().item()

                val_epoch_loss = val_loss / len(val_dataset)
                val_epoch_acc = val_correct / val_total
                
                # ログが多すぎる場合は適宜コメントアウトしてください
                # print(f'Fold {fold+1} Ep {epoch+1} | T_Acc:{epoch_acc:.3f} V_Acc:{val_epoch_acc:.3f}')

                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc)
                history['val_loss'].append(val_epoch_loss)
                history['val_acc'].append(val_epoch_acc)

            # 保存 (モデル・グラフ)
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, f'resnet18_fold{fold+1}.pth'))

            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(history['train_loss'], label='Train')
            plt.plot(history['val_loss'], label='Val')
            plt.title(f'Loss (TotalSize={ACTUAL_TOTAL})')
            plt.legend()
            plt.grid()

            plt.subplot(1, 2, 2)
            plt.plot(history['train_acc'], label='Train')
            plt.plot(history['val_acc'], label='Val')
            plt.title(f'Accuracy (TotalSize={ACTUAL_TOTAL})')
            plt.legend()
            plt.grid()

            plt.savefig(os.path.join(SAVE_DIR, f'curve_fold{fold+1}.png'))
            plt.close()
        
        print(f"-> {total_size_req}枚の実験完了。保存先: {SAVE_DIR}")

    print("\n全てのパターンの学習が完了しました。")

if __name__ == '__main__':
    main()

使用デバイス: cpu
検知されたクラス: ['A', 'B', 'C']

★ 実験開始: 目標データ総数 18枚
【設定確認】
  指定: 18枚 -> 実使用: 18枚 (各クラス 6枚)
  保存先: resnet18_cv_total18

--- データセットの抽出 ---
実際に使用するデータ総数: 18枚

--- [Total: 18] Fold 1/6 ---

--- [Total: 18] Fold 2/6 ---

--- [Total: 18] Fold 3/6 ---

--- [Total: 18] Fold 4/6 ---

--- [Total: 18] Fold 5/6 ---

--- [Total: 18] Fold 6/6 ---
-> 18枚の実験完了。保存先: resnet18_cv_total18

★ 実験開始: 目標データ総数 30枚
【設定確認】
  指定: 30枚 -> 実使用: 30枚 (各クラス 10枚)
  保存先: resnet18_cv_total30

--- データセットの抽出 ---
実際に使用するデータ総数: 30枚

--- [Total: 30] Fold 1/6 ---

--- [Total: 30] Fold 2/6 ---

--- [Total: 30] Fold 3/6 ---

--- [Total: 30] Fold 4/6 ---

--- [Total: 30] Fold 5/6 ---

--- [Total: 30] Fold 6/6 ---
-> 30枚の実験完了。保存先: resnet18_cv_total30

★ 実験開始: 目標データ総数 42枚
【設定確認】
  指定: 42枚 -> 実使用: 42枚 (各クラス 14枚)
  保存先: resnet18_cv_total42

--- データセットの抽出 ---
実際に使用するデータ総数: 42枚

--- [Total: 42] Fold 1/6 ---

--- [Total: 42] Fold 2/6 ---

--- [Total: 42] Fold 3/6 ---

--- [Total: 42] Fold 4/6 ---

--- [Total: 42] Fold 5/6 ---

---

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def main():
    # -----------------------------
    # 1. 共通設定パラメータ (学習時と一致させる)
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 8
    NUM_CLASSES = 3
    K_FOLDS = 6
    
    # ★学習プログラムと同じリストを指定してください
    TARGET_TOTAL_SIZES = [18, 30, 42, 54, 66,72, 84, 96, 108, 120, 132, 144, 156] 

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データセット読み込み (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    print(f"検知されたクラス: {class_names}")

    # 全実験のまとめ用リスト
    global_comparison = []

    # =========================================================
    # ★ループ開始：指定したデータサイズごとに評価
    # =========================================================
    for total_size_req in TARGET_TOTAL_SIZES:
        # パラメータ計算 (学習時と同じロジック)
        IMAGES_PER_CLASS = total_size_req // NUM_CLASSES
        ACTUAL_TOTAL = IMAGES_PER_CLASS * NUM_CLASSES
        SEED_VALUE = total_size_req # ★重要: 学習時と同じシード計算式

        SAVE_DIR = f'resnet18_cv_total{ACTUAL_TOTAL}'
        
        print(f"\n========================================================")
        print(f"★ 評価開始: 目標 {total_size_req}枚 (実数 {ACTUAL_TOTAL}枚)")
        print(f"   読込先: {SAVE_DIR}")
        print(f"========================================================")

        if not os.path.exists(SAVE_DIR):
            print(f"エラー: フォルダ {SAVE_DIR} が見つかりません。スキップします。")
            continue

        # -----------------------------
        # 3. データセット抽出の再現
        # -----------------------------
        used_indices = []
        used_labels = []

        # ★学習時と同じシードで固定して再現
        np.random.seed(SEED_VALUE) 

        for class_idx in range(NUM_CLASSES):
            indices_of_class = np.where(all_targets == class_idx)[0]
            
            if len(indices_of_class) < IMAGES_PER_CLASS:
                 select_num = len(indices_of_class)
            else:
                 select_num = IMAGES_PER_CLASS

            np.random.shuffle(indices_of_class)
            selected_indices = indices_of_class[:select_num]
            
            used_indices.extend(selected_indices)
            used_labels.extend([class_idx] * len(selected_indices))

        used_indices = np.array(used_indices)
        used_labels = np.array(used_labels)

        # -----------------------------
        # 4. K-Fold 評価実行
        # -----------------------------
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
        
        fold_results_summary = []

        for fold, (train_idx_in_subset, val_idx_in_subset) in enumerate(skf.split(used_indices, used_labels)):
            fold_num = fold + 1
            # print(f" - Fold {fold_num}/{K_FOLDS} 評価中...")

            # 評価データ抽出
            val_real_indices = used_indices[val_idx_in_subset]
            val_dataset = Subset(full_dataset, val_real_indices)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

            # モデル準備
            model = models.resnet18(weights=None)
            model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
            
            weight_path = os.path.join(SAVE_DIR, f'resnet18_fold{fold_num}.pth')
            if not os.path.exists(weight_path):
                print(f"   警告: モデル {weight_path} なし。スキップ。")
                continue

            model.load_state_dict(torch.load(weight_path, map_location=device))
            model = model.to(device)
            model.eval()

            # 推論
            all_preds = []
            all_labels = []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs = inputs.to(device)
                    outputs = model(inputs)
                    _, predicted = torch.max(outputs, 1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            # 指標計算
            acc = accuracy_score(all_labels, all_preds)
            report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)

            # 結果格納
            fold_res = {'Fold': fold_num, 'Accuracy': acc, 'Macro_F1': report_dict['macro avg']['f1-score']}
            for cls_name in class_names:
                fold_res[f'{cls_name}_Prec'] = report_dict[cls_name]['precision']
                fold_res[f'{cls_name}_Recall'] = report_dict[cls_name]['recall']
            fold_results_summary.append(fold_res)

            # 混同行列の保存
            cm = confusion_matrix(all_labels, all_preds)
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            
            # 画像保存 (Foldごとの詳細は枚数が多すぎるならコメントアウト可)
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
            plt.title(f'Total {ACTUAL_TOTAL} - Fold {fold_num}\n(Acc: {acc:.2%})')
            plt.ylabel('True')
            plt.xlabel('Pred')
            plt.savefig(os.path.join(SAVE_DIR, f'confusion_matrix_fold{fold_num}.png'), bbox_inches='tight')
            plt.close()
            
            # CSV保存
            cm_df.to_csv(os.path.join(SAVE_DIR, f'confusion_matrix_fold{fold_num}.csv'))

        # -----------------------------
        # 5. このデータサイズのまとめCSV出力
        # -----------------------------
        if fold_results_summary:
            df_summary = pd.DataFrame(fold_results_summary)
            mean_row = df_summary.mean(numeric_only=True)
            mean_row['Fold'] = 'Average'
            df_summary = pd.concat([df_summary, mean_row.to_frame().T], ignore_index=True)
            
            # 順番整理
            cols = ['Fold'] + [c for c in df_summary.columns if c != 'Fold']
            df_summary = df_summary[cols]
            
            csv_path = os.path.join(SAVE_DIR, 'evaluation_summary.csv')
            df_summary.to_csv(csv_path, index=False, float_format='%.4f')
            
            print(f" -> 平均正解率: {mean_row['Accuracy']:.4f}")

            # グローバル比較用に追加
            global_comparison.append({
                'Total_Images': ACTUAL_TOTAL,
                'Requested_Images': total_size_req,
                'Average_Accuracy': mean_row['Accuracy'],
                'Average_Macro_F1': mean_row['Macro_F1']
            })

    # =========================================================
    # 6. 全実験の比較ファイル出力
    # =========================================================
    print("\n========================================================")
    print(" 全パターンの評価完了")
    print("========================================================")
    
    if global_comparison:
        df_global = pd.DataFrame(global_comparison)
        df_global.to_csv('final_size_accuracy_comparison.csv', index=False, float_format='%.4f')
        print("★ データ数ごとの精度推移を 'final_size_accuracy_comparison.csv' に保存しました。")
        print(df_global)

if __name__ == '__main__':
    main()

使用デバイス: cpu
検知されたクラス: ['A', 'B', 'C']

★ 評価開始: 目標 18枚 (実数 18枚)
   読込先: resnet18_cv_total18


/tmp/ipykernel_612145/3643012662.py:164: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 平均正解率: 0.5000

★ 評価開始: 目標 30枚 (実数 30枚)
   読込先: resnet18_cv_total30


/tmp/ipykernel_612145/3643012662.py:164: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 平均正解率: 0.5333

★ 評価開始: 目標 42枚 (実数 42枚)
   読込先: resnet18_cv_total42


...

/tmp/ipykernel_612145/3643012662.py:164: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 平均正解率: 0.6364

★ 評価開始: 目標 144枚 (実数 144枚)
   読込先: resnet18_cv_total144


/tmp/ipykernel_612145/3643012662.py:164: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 平均正解率: 0.6181

★ 評価開始: 目標 156枚 (実数 156枚)
   読込先: resnet18_cv_total156
 -> 平均正解率: 0.6218

 全パターンの評価完了
★ データ数ごとの精度推移を 'final_size_accuracy_comparison.csv' に保存しました。
    Total_Images  Requested_Images  Average_Accuracy  Average_Macro_F1
0             18                18          0.500000          0.379630
1             30                30          0.533333          0.490741
2             42                42          0.619048          0.518783
3             54                54          0.703704          0.680423
4             72                72          0.597222          0.525794
5             84                84          0.630952          0.612414
6             96                96          0.687500          0.655979
7            108               108          0.648148          0.631895
8            120               120          0.600000          0.587690
9            132               132          0.636364          0.613648
10           144               144          0.618056 

/tmp/ipykernel_612145/3643012662.py:164: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def main():
    # -----------------------------
    # 1. 共通設定パラメータ (学習時と一致させる)
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 8
    NUM_CLASSES = 3
    K_FOLDS = 6
    
    # ★学習プログラムと同じリストを指定してください
    TARGET_TOTAL_SIZES = [18, 30, 42, 54, 72, 84, 96, 108, 120, 132, 144, 156] 

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データセット読み込み (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    print(f"検知されたクラス: {class_names}")

    # ★追加: 全画像のファイルパスを取得 (インデックスとファイル名の紐付け用)
    # full_dataset.samples は (path, class_index) のリスト
    all_file_paths = [s[0] for s in full_dataset.samples]

    # 全実験のまとめ用リスト
    global_comparison = []

    # =========================================================
    # ★ループ開始：指定したデータサイズごとに評価
    # =========================================================
    for total_size_req in TARGET_TOTAL_SIZES:
        # パラメータ計算 (学習時と同じロジック)
        IMAGES_PER_CLASS = total_size_req // NUM_CLASSES
        ACTUAL_TOTAL = IMAGES_PER_CLASS * NUM_CLASSES
        SEED_VALUE = total_size_req # ★重要: 学習時と同じシード計算式

        SAVE_DIR = f'resnet18_cv_total{ACTUAL_TOTAL}'
        
        print(f"\n========================================================")
        print(f"★ 評価開始: 目標 {total_size_req}枚 (実数 {ACTUAL_TOTAL}枚)")
        print(f"   読込先: {SAVE_DIR}")
        print(f"========================================================")

        if not os.path.exists(SAVE_DIR):
            print(f"エラー: フォルダ {SAVE_DIR} が見つかりません。スキップします。")
            continue

        # -----------------------------
        # 3. データセット抽出の再現
        # -----------------------------
        used_indices = []
        used_labels = []

        # ★学習時と同じシードで固定して再現
        np.random.seed(SEED_VALUE) 

        for class_idx in range(NUM_CLASSES):
            indices_of_class = np.where(all_targets == class_idx)[0]
            
            if len(indices_of_class) < IMAGES_PER_CLASS:
                 select_num = len(indices_of_class)
            else:
                 select_num = IMAGES_PER_CLASS

            np.random.shuffle(indices_of_class)
            selected_indices = indices_of_class[:select_num]
            
            used_indices.extend(selected_indices)
            used_labels.extend([class_idx] * len(selected_indices))

        used_indices = np.array(used_indices)
        used_labels = np.array(used_labels)

        # -----------------------------
        # 4. K-Fold 評価実行
        # -----------------------------
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
        
        fold_results_summary = []
        
        # ★追加: このデータサイズにおける全詳細ログ
        size_specific_details = []

        for fold, (train_idx_in_subset, val_idx_in_subset) in enumerate(skf.split(used_indices, used_labels)):
            fold_num = fold + 1
            
            # 評価データ抽出
            val_real_indices = used_indices[val_idx_in_subset]
            val_dataset = Subset(full_dataset, val_real_indices)
            
            # shuffle=False は必須 (インデックス順序を保つため)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

            # モデル準備
            model = models.resnet18(weights=None)
            model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
            
            weight_path = os.path.join(SAVE_DIR, f'resnet18_fold{fold_num}.pth')
            if not os.path.exists(weight_path):
                print(f"   警告: モデル {weight_path} なし。スキップ。")
                continue

            model.load_state_dict(torch.load(weight_path, map_location=device))
            model = model.to(device)
            model.eval()

            # 推論
            all_preds = []
            all_labels = []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs = inputs.to(device)
                    outputs = model(inputs)
                    _, predicted = torch.max(outputs, 1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            # ----------- ★追加機能: ファイル名と予測詳細の記録 -----------
            # このFoldで使われたファイル名を取得
            val_filenames = [os.path.basename(all_file_paths[i]) for i in val_real_indices]
            
            # DataFrame作成
            fold_detail_df = pd.DataFrame({
                'Total_Images': ACTUAL_TOTAL,
                'Fold': fold_num,
                'filename': val_filenames,
                'True_Label': all_labels,
                'Predicted_Label': all_preds,
                'Is_Correct': [t == p for t, p in zip(all_labels, all_preds)]
            })
            size_specific_details.append(fold_detail_df)
            # ---------------------------------------------------------

            # 指標計算
            acc = accuracy_score(all_labels, all_preds)
            report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)

            # 結果格納
            fold_res = {'Fold': fold_num, 'Accuracy': acc, 'Macro_F1': report_dict['macro avg']['f1-score']}
            for cls_name in class_names:
                fold_res[f'{cls_name}_Prec'] = report_dict[cls_name]['precision']
                fold_res[f'{cls_name}_Recall'] = report_dict[cls_name]['recall']
            fold_results_summary.append(fold_res)

            # 混同行列の保存
            cm = confusion_matrix(all_labels, all_preds)
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
            plt.title(f'Total {ACTUAL_TOTAL} - Fold {fold_num}\n(Acc: {acc:.2%})')
            plt.ylabel('True')
            plt.xlabel('Pred')
            plt.savefig(os.path.join(SAVE_DIR, f'confusion_matrix_fold{fold_num}.png'), bbox_inches='tight')
            plt.close()
            
            cm_df.to_csv(os.path.join(SAVE_DIR, f'confusion_matrix_fold{fold_num}.csv'))

        # -----------------------------
        # 5. このデータサイズのまとめCSV出力
        # -----------------------------
        if fold_results_summary:
            # サマリーCSV
            df_summary = pd.DataFrame(fold_results_summary)
            mean_row = df_summary.mean(numeric_only=True)
            mean_row['Fold'] = 'Average'
            df_summary = pd.concat([df_summary, mean_row.to_frame().T], ignore_index=True)
            
            cols = ['Fold'] + [c for c in df_summary.columns if c != 'Fold']
            df_summary = df_summary[cols]
            
            csv_path = os.path.join(SAVE_DIR, 'evaluation_summary.csv')
            df_summary.to_csv(csv_path, index=False, float_format='%.4f')
            
            # ★追加: 詳細データCSV (ファイル名付き)
            if size_specific_details:
                df_details = pd.concat(size_specific_details, ignore_index=True)
                detail_path = os.path.join(SAVE_DIR, 'resnet_evaluation_details.csv')
                df_details.to_csv(detail_path, index=False)
                print(f" -> 詳細ログ保存: {detail_path}")
            
            print(f" -> 平均正解率: {mean_row['Accuracy']:.4f}")

            # グローバル比較用に追加
            global_comparison.append({
                'Total_Images': ACTUAL_TOTAL,
                'Requested_Images': total_size_req,
                'Average_Accuracy': mean_row['Accuracy'],
                'Average_Macro_F1': mean_row['Macro_F1']
            })

    # =========================================================
    # 6. 全実験の比較ファイル出力
    # =========================================================
    print("\n========================================================")
    print(" 全パターンの評価完了")
    print("========================================================")
    
    if global_comparison:
        df_global = pd.DataFrame(global_comparison)
        df_global.to_csv('final_size_accuracy_comparison.csv', index=False, float_format='%.4f')
        print("★ データ数ごとの精度推移を 'final_size_accuracy_comparison.csv' に保存しました。")
        print(df_global)

if __name__ == '__main__':
    main()

使用デバイス: cpu
検知されたクラス: ['A', 'B', 'C']

★ 評価開始: 目標 18枚 (実数 18枚)
   読込先: resnet18_cv_total18


/tmp/ipykernel_612145/2394143620.py:187: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total18/resnet_evaluation_details.csv
 -> 平均正解率: 0.5000

★ 評価開始: 目標 30枚 (実数 30枚)
   読込先: resnet18_cv_total30


/tmp/ipykernel_612145/2394143620.py:187: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total30/resnet_evaluation_details.csv
 -> 平均正解率: 0.5333

★ 評価開始: 目標 42枚 (実数 42枚)
   読込先: resnet18_cv_total42


...

/tmp/ipykernel_612145/2394143620.py:187: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total132/resnet_evaluation_details.csv
 -> 平均正解率: 0.6364

★ 評価開始: 目標 144枚 (実数 144枚)
   読込先: resnet18_cv_total144


/tmp/ipykernel_612145/2394143620.py:187: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total144/resnet_evaluation_details.csv
 -> 平均正解率: 0.6181

★ 評価開始: 目標 156枚 (実数 156枚)
   読込先: resnet18_cv_total156
 -> 詳細ログ保存: resnet18_cv_total156/resnet_evaluation_details.csv
 -> 平均正解率: 0.6218

 全パターンの評価完了
★ データ数ごとの精度推移を 'final_size_accuracy_comparison.csv' に保存しました。
    Total_Images  Requested_Images  Average_Accuracy  Average_Macro_F1
0             18                18          0.500000          0.379630
1             30                30          0.533333          0.490741
2             42                42          0.619048          0.518783
3             54                54          0.703704          0.680423
4             72                72          0.597222          0.525794
5             84                84          0.630952          0.612414
6             96                96          0.687500          0.655979
7            108               108          0.648148          0.631895
8            120               120          0.600000          0.58769

/tmp/ipykernel_612145/2394143620.py:187: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


In [4]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def main():
    # -----------------------------
    # 1. 共通設定パラメータ (学習時と一致させる)
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 8
    NUM_CLASSES = 3
    K_FOLDS = 6
    
    # ★学習プログラムと同じリストを指定してください
    TARGET_TOTAL_SIZES = [18, 30, 42, 54, 72, 84, 96, 108, 120, 132, 144, 156] 

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データセット読み込み (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    print(f"検知されたクラス: {class_names}")

    # ★追加: 全画像のファイルパスを取得
    all_file_paths = [s[0] for s in full_dataset.samples]

    # 全実験のまとめ用リスト
    global_comparison = []

    # =========================================================
    # ★ループ開始：指定したデータサイズごとに評価
    # =========================================================
    for total_size_req in TARGET_TOTAL_SIZES:
        # パラメータ計算 (学習時と同じロジック)
        IMAGES_PER_CLASS = total_size_req // NUM_CLASSES
        ACTUAL_TOTAL = IMAGES_PER_CLASS * NUM_CLASSES
        SEED_VALUE = total_size_req # ★重要: 学習時と同じシード計算式

        SAVE_DIR = f'resnet18_cv_total{ACTUAL_TOTAL}'
        
        print(f"\n========================================================")
        print(f"★ 評価開始: 目標 {total_size_req}枚 (実数 {ACTUAL_TOTAL}枚)")
        print(f"   読込先: {SAVE_DIR}")
        print(f"========================================================")

        if not os.path.exists(SAVE_DIR):
            print(f"エラー: フォルダ {SAVE_DIR} が見つかりません。スキップします。")
            continue

        # -----------------------------
        # 3. データセット抽出の再現
        # -----------------------------
        used_indices = []
        used_labels = []

        # ★学習時と同じシードで固定して再現
        np.random.seed(SEED_VALUE) 

        for class_idx in range(NUM_CLASSES):
            indices_of_class = np.where(all_targets == class_idx)[0]
            
            if len(indices_of_class) < IMAGES_PER_CLASS:
                 select_num = len(indices_of_class)
            else:
                 select_num = IMAGES_PER_CLASS

            np.random.shuffle(indices_of_class)
            selected_indices = indices_of_class[:select_num]
            
            used_indices.extend(selected_indices)
            used_labels.extend([class_idx] * len(selected_indices))

        used_indices = np.array(used_indices)
        used_labels = np.array(used_labels)

        # -----------------------------
        # 4. K-Fold 評価実行
        # -----------------------------
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
        
        fold_results_summary = []
        
        # ★追加: このデータサイズにおける全詳細ログ
        size_specific_details = []

        for fold, (train_idx_in_subset, val_idx_in_subset) in enumerate(skf.split(used_indices, used_labels)):
            fold_num = fold + 1
            
            # 評価データ抽出
            val_real_indices = used_indices[val_idx_in_subset]
            val_dataset = Subset(full_dataset, val_real_indices)
            
            # shuffle=False は必須 (インデックス順序を保つため)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

            # モデル準備
            model = models.resnet18(weights=None)
            model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
            
            weight_path = os.path.join(SAVE_DIR, f'resnet18_fold{fold_num}.pth')
            if not os.path.exists(weight_path):
                print(f"   警告: モデル {weight_path} なし。スキップ。")
                continue

            model.load_state_dict(torch.load(weight_path, map_location=device))
            model = model.to(device)
            model.eval()

            # 推論
            all_preds = []
            all_labels = []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs = inputs.to(device)
                    outputs = model(inputs)
                    _, predicted = torch.max(outputs, 1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            # ----------- ★追加機能: ファイル名と予測詳細の記録 -----------
            val_filenames = [os.path.basename(all_file_paths[i]) for i in val_real_indices]
            
            fold_detail_df = pd.DataFrame({
                'Total_Images': ACTUAL_TOTAL,
                'Fold': fold_num,
                'filename': val_filenames,
                'True_Label': all_labels,
                'Predicted_Label': all_preds,
                'Is_Correct': [t == p for t, p in zip(all_labels, all_preds)]
            })
            size_specific_details.append(fold_detail_df)
            # ---------------------------------------------------------

            # 指標計算
            acc = accuracy_score(all_labels, all_preds)
            report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)

            fold_res = {'Fold': fold_num, 'Accuracy': acc, 'Macro_F1': report_dict['macro avg']['f1-score']}
            for cls_name in class_names:
                fold_res[f'{cls_name}_Prec'] = report_dict[cls_name]['precision']
                fold_res[f'{cls_name}_Recall'] = report_dict[cls_name]['recall']
            fold_results_summary.append(fold_res)

            # 混同行列の保存
            cm = confusion_matrix(all_labels, all_preds)
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
            plt.title(f'Total {ACTUAL_TOTAL} - Fold {fold_num}\n(Acc: {acc:.2%})')
            plt.ylabel('True')
            plt.xlabel('Pred')
            plt.savefig(os.path.join(SAVE_DIR, f'confusion_matrix_fold{fold_num}.png'), bbox_inches='tight')
            plt.close()
            
            cm_df.to_csv(os.path.join(SAVE_DIR, f'confusion_matrix_fold{fold_num}.csv'))

        # -----------------------------
        # 5. このデータサイズのまとめCSV出力
        # -----------------------------
        if fold_results_summary:
            # サマリーCSV
            df_summary = pd.DataFrame(fold_results_summary)
            mean_row = df_summary.mean(numeric_only=True)
            mean_row['Fold'] = 'Average'
            df_summary = pd.concat([df_summary, mean_row.to_frame().T], ignore_index=True)
            
            cols = ['Fold'] + [c for c in df_summary.columns if c != 'Fold']
            df_summary = df_summary[cols]
            
            csv_path = os.path.join(SAVE_DIR, 'evaluation_summary.csv')
            df_summary.to_csv(csv_path, index=False, float_format='%.4f')
            
            # ★詳細データCSV (ファイル名付き)
            if size_specific_details:
                df_details = pd.concat(size_specific_details, ignore_index=True)
                detail_path = os.path.join(SAVE_DIR, 'resnet_evaluation_details.csv')
                df_details.to_csv(detail_path, index=False)
                print(f" -> 詳細ログ保存: {detail_path}")
            
            print(f" -> 平均正解率: {mean_row['Accuracy']:.4f}")

            # グローバル比較用に追加
            global_comparison.append({
                'Total_Images': ACTUAL_TOTAL,
                'Requested_Images': total_size_req,
                'Average_Accuracy': mean_row['Accuracy'],
                'Average_Macro_F1': mean_row['Macro_F1']
            })

    # =========================================================
    # 6. 全実験の比較ファイル出力・グラフ化
    # =========================================================
    print("\n========================================================")
    print(" 全パターンの評価完了")
    print("========================================================")
    
    if global_comparison:
        df_global = pd.DataFrame(global_comparison)
        csv_save_path = 'final_size_accuracy_comparison.csv'
        df_global.to_csv(csv_save_path, index=False, float_format='%.4f')
        print(f"★ データ数ごとの精度推移を '{csv_save_path}' に保存しました。")
        print(df_global)

        # ----------- ★追加機能: グラフ作成 -----------
        plt.figure(figsize=(10, 6))
        
        # 折れ線グラフのプロット
        plt.plot(df_global['Total_Images'], df_global['Average_Accuracy'], marker='o', linestyle='-', color='b', label='ResNet18')
        
        # タイトルと軸ラベル
        plt.title('Accuracy vs Data Size (ResNet18)', fontsize=14)
        plt.xlabel('Number of Images', fontsize=12)
        plt.ylabel('Average Accuracy', fontsize=12)
        
        # グリッドと凡例
        plt.grid(True)
        plt.legend()
        
        # 軸の範囲設定 (0.0〜1.05)
        plt.ylim(0, 1.05)
        
        # 画像保存
        graph_save_path = 'final_size_accuracy_plot.png'
        plt.savefig(graph_save_path)
        print(f"★ 精度推移グラフを '{graph_save_path}' に保存しました。")
        plt.close()

if __name__ == '__main__':
    main()

使用デバイス: cpu
検知されたクラス: ['A', 'B', 'C']

★ 評価開始: 目標 18枚 (実数 18枚)
   読込先: resnet18_cv_total18


/tmp/ipykernel_612145/1174444276.py:183: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total18/resnet_evaluation_details.csv
 -> 平均正解率: 0.5000

★ 評価開始: 目標 30枚 (実数 30枚)
   読込先: resnet18_cv_total30


/tmp/ipykernel_612145/1174444276.py:183: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total30/resnet_evaluation_details.csv
 -> 平均正解率: 0.5333

★ 評価開始: 目標 42枚 (実数 42枚)
   読込先: resnet18_cv_total42


...

/tmp/ipykernel_612145/1174444276.py:183: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total132/resnet_evaluation_details.csv
 -> 平均正解率: 0.6364

★ 評価開始: 目標 144枚 (実数 144枚)
   読込先: resnet18_cv_total144


/tmp/ipykernel_612145/1174444276.py:183: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'


 -> 詳細ログ保存: resnet18_cv_total144/resnet_evaluation_details.csv
 -> 平均正解率: 0.6181

★ 評価開始: 目標 156枚 (実数 156枚)
   読込先: resnet18_cv_total156
 -> 詳細ログ保存: resnet18_cv_total156/resnet_evaluation_details.csv
 -> 平均正解率: 0.6218

 全パターンの評価完了
★ データ数ごとの精度推移を 'final_size_accuracy_comparison.csv' に保存しました。
    Total_Images  Requested_Images  Average_Accuracy  Average_Macro_F1
0             18                18          0.500000          0.379630
1             30                30          0.533333          0.490741
2             42                42          0.619048          0.518783
3             54                54          0.703704          0.680423
4             72                72          0.597222          0.525794
5             84                84          0.630952          0.612414
6             96                96          0.687500          0.655979
7            108               108          0.648148          0.631895
8            120               120          0.600000          0.58769

/tmp/ipykernel_612145/1174444276.py:183: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Average' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mean_row['Fold'] = 'Average'
